# 🧠 Deep Learning Time-Series Training (CNN & MLP)

Notebook này được sử dụng để huấn luyện 2 mô hình Deep Learning (1D-CNN và Multi-Layer Perceptron) bằng PyTorch.
**Điểm đặc biệt đối với Mạng Nơ-ron:**
- Mạng Nơ-ron không có khả năng hiểu giá trị `NaN` như các thuật toán Cây (LightGBM). Vì vậy bắt buộc phải có bước Pre-processing (FillNA, Clip Infinity).
- Mạng Nơ-ron rất nhạy cảm với thang đo của features. Bắt buộc phải được chuẩn hóa qua hệ thống `StandardScaler`.
- Các mô hình này được train qua chiến lược Time-Series K-Fold.

In [ ]:
import os
import sys
import gc
import numpy as np
import pandas as pd
import warnings
import torch
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
sys.path.append(os.path.abspath('..'))

from src.training.cv_splitter import TimeSeriesCVSplitter
from src.training.metrics import rmspe
from src.models.cnn_model import CNN1DModel
from src.models.mlp_model import MLPModel
from src.training.trainer import NNTrainer

### 1. Nạp Dữ Liệu và Tiền Xử Lý (Imputation & Scaling)

In [ ]:
FEATURES_PATH = '/kaggle/input/notebooks/thnhvinhs/notebook22d94af6ba/Realized-Volatility-Prediction/data/processed/features_FINAL.feather'

print("📊 Đang nạp tập dữ liệu đặc trưng toàn cục...")
df = pd.read_feather(FEATURES_PATH)
print(f"Kích thước ma trận dữ liệu: {df.shape}")

# 1. Khôi phục trục dòng thời gian thực tế
splitter = TimeSeriesCVSplitter(n_splits=4, random_state=42)
if 'true_time_id' not in df.columns:
    print("⏳ Đang tính toán t-SNE để tái cấu trúc trục thời gian...")
    time_order = splitter.reverse_engineer_time_order(df, df)
    df = df.merge(time_order, on='time_id', how='left')

# 2. Phân định danh sách đặc trưng học tập
not_features = ['row_id', 'time_id', 'stock_id', 'target', 'true_time_id', 'tsne_val']
features = [col for col in df.columns if col not in not_features]

print(f"🛠️ Đang xử lý làm sạch dữ liệu chuyên sâu cho Mạng Nơ-ron...")
# Xử lý các giá trị vô cùng (nếu có) sinh ra từ các phép chia đặc trưng vi cấu trúc sổ lệnh
df[features] = df[features].replace([np.inf, -np.inf], np.nan)

# Điền các giá trị trống bằng giá trị trung bình của từng cột đặc trưng tương ứng
# Sử dụng phương pháp gán giá trị nhanh, tối ưu hóa vùng nhớ
df[features] = df[features].fillna(df[features].mean())

print(f"✅ Dữ liệu đã được làm sạch 100%. Sẵn sàng đưa vào không gian Tensor.")

### 2. Vòng lặp Huấn Luyện (Training Loop)

In [ ]:
# Mảng lưu trữ kết quả dự đoán Out-of-Fold (OOF)
cnn_oof_preds = np.zeros(len(df))
mlp_oof_preds = np.zeros(len(df))

# Khởi tạo thư mục
os.makedirs('./models/cnn', exist_ok=True)
os.makedirs('./models/mlp', exist_ok=True)

EPOCHS = 40
BATCH_SIZE = 1024

folds_generator = splitter.split(df)

for fold, (train_idx, val_idx) in enumerate(folds_generator):
    print(f"\n" + "="*40 + f" 🧠 HUẤN LUYỆN PYTORCH FOLD {fold + 1}/4 " + "="*40)
    
    # Tách dữ liệu
    X_train_raw = df.loc[train_idx, features].values
    y_train_raw = df.loc[train_idx, 'target'].values
    X_val_raw = df.loc[val_idx, features].values
    y_val_raw = df.loc[val_idx, 'target'].values
    
    # 🔒 CHỐNG DATA LEAKAGE: Áp dụng Scaler độc lập
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw)
    X_val_scaled = scaler.transform(X_val_raw)
    
    input_dim = len(features)
    
    # ------------------ TIẾN HÀNH HUẤN LUYỆN 1D-CNN ------------------
    print("   ↳ 🎥 Đang kích hoạt NNTrainer cho kiến trúc 1D-CNN...")
    cnn_model = CNN1DModel(num_features=input_dim)
    cnn_trainer = NNTrainer(model=cnn_model, device='auto', lr=1e-3, weight_decay=1e-5)
    
    # Gọi hàm train 1 dòng duy nhất nhờ sự đóng gói của Agent
    best_cnn_loss = cnn_trainer.train(
        X_train_scaled, y_train_raw, 
        X_val_scaled, y_val_raw, 
        batch_size=BATCH_SIZE, epochs=EPOCHS, patience=5
    )
    
    # Suy luận và lưu model
    cnn_oof_preds[val_idx] = cnn_trainer.predict(X_val_scaled, batch_size=BATCH_SIZE)
    cnn_trainer.save_model(f'./models/cnn/cnn_fold{fold+1}.pth')
    
    # ------------------ TIẾN HÀNH HUẤN LUYỆN MLP ------------------
    print("\n   ↳ ⚡ Đang kích hoạt NNTrainer cho kiến trúc MLP...")
    mlp_model = MLPModel(num_features=input_dim) # Lưu ý: Kiểm tra file mlp_model.py xem tham số khởi tạo là num_features hay input_dim
    mlp_trainer = NNTrainer(model=mlp_model, device='auto', lr=1e-3, weight_decay=1e-5)
    
    best_mlp_loss = mlp_trainer.train(
        X_train_scaled, y_train_raw, 
        X_val_scaled, y_val_raw, 
        batch_size=BATCH_SIZE, epochs=EPOCHS, patience=5
    )
    
    # Suy luận và lưu model
    mlp_oof_preds[val_idx] = mlp_trainer.predict(X_val_scaled, batch_size=BATCH_SIZE)
    mlp_trainer.save_model(f'./models/mlp/mlp_fold{fold+1}.pth')
    
    # Giải phóng RAM
    del X_train_scaled, X_val_scaled, cnn_model, cnn_trainer, mlp_model, mlp_trainer
    gc.collect()

### 3. Tổng Hợp Kết Quả & Đánh Giá Ensembling

In [ ]:
valid_indexes = cnn_oof_preds > 0
y_true_final = df.loc[valid_indexes, 'target'].values

final_cnn_rmspe = rmspe(y_true_final, cnn_oof_preds[valid_indexes])
final_mlp_rmspe = rmspe(y_true_final, mlp_oof_preds[valid_indexes])

# Ensemble CNN và MLP theo tỷ lệ 50/50 để xem sức mạnh của sự kết hợp
nn_ensemble_preds = (cnn_oof_preds[valid_indexes] + mlp_oof_preds[valid_indexes]) / 2.0
final_ensemble_rmspe = rmspe(y_true_final, nn_ensemble_preds)

print("="*50)
print(f"🔥 ĐIỂM SỐ CHUNG CUỘC CNN 1D      : {final_cnn_rmspe:.5f}")
print(f"🔥 ĐIỂM SỐ CHUNG CUỘC MLP         : {final_mlp_rmspe:.5f}")
print(f"🌟 ENSEMBLE CNN + MLP (50/50)    : {final_ensemble_rmspe:.5f}")
print("="*50)